# Variably saturated flow (VSF): putting Richards into the valley

The synthetic valley so far treats the soil above the water table in a simple way. In the [`synthetic-valley-a-base`](synthetic-valley-a-base.ipynb) model recharge lands straight on the water table, and in the [`synthetic-valley-b-advanced`](synthetic-valley-b-advanced.ipynb) model the **UZF** package routes it through the unsaturated zone with a kinematic-wave approximation. The **VSF** package replaces that with **Richards' equation**: the pressure head is solved in every cell, so the soil has a moisture profile, a capillary fringe, and a lag between recharge and the water table.

This notebook asks a practical question: **what does it take to put the VSF package into the valley model we already have, and what is different?** We start from the base valley (recharge, river, coastal general-head boundary, and wells) because it is the simplest model to change, and work through the steps one at a time.

> **Package reference.** VSF is the Richards-based unsaturated-zone flow package for MODFLOW 6, added through `flopy.mf6.ModflowGwfvsf`. The soil-characteristic curves use the van Genuchten (1980) parameters.

## What this notebook covers

By the end of this notebook you will be able to:

- see why the **vertical grid** has to resolve the unsaturated zone, and what VSF does if it does not,
- **refine** the top layer of the valley and re-attach the boundaries and wells to the new layers,
- list what the **VSF package** needs: cells flagged for Richards flow, soil parameters, non-convertible cells, Newton, and a robust solver, and
- check how much the answer depends on the **cell thickness** and on the **time step**.

## Imports

Import FloPy, Matplotlib, and NumPy. `vsf_helpers` holds the van Genuchten soil data and curves, `load_valley` that loads the base valley, `refine_top_layer` that refines the grid and moves everything onto it, and a reader for the water table.

In [ ]:
%matplotlib inline
import pathlib as pl
import time

import flopy
import matplotlib.pyplot as plt
import numpy as np
import vsf_helpers as vh
from vsf_helpers import SOILS

width = 17.15 / 2.54  # figure width, inches, as in the base notebook

## Step 0 &mdash; The starting point: the base valley

`load_valley()` loads the base valley model and sets its workspace. It leaves out the lake evapotranspiration and the observation wells, which are not needed here. Recharge, the stream (RIV), the coast (GHB), and the two pumping wells stay.

Run it as it is. The model is saturated only: recharge goes straight into the cells of layer 1 and the water table is the head in that layer. This is the reference the VSF models are compared with. The base model takes one time step per yearly stress period; `set_substeps()` splits each yearly period into `nstp` steps (the steady-state period keeps one), because Richards' equation needs smaller steps. Every model here uses the same steps, so only the physics differs.

In [ ]:
nstp = 4  # time steps per yearly stress period, for every model below

base_ws = pl.Path("models/synthetic-valley-vsf-base")
sim_base, gwf_base = vh.load_valley(base_ws)
vh.set_substeps(sim_base, nstp)


def run(sim):
    """Write and run a simulation, and return the run time in seconds."""
    sim.write_simulation(silent=True)
    t0 = time.time()
    success, buff = sim.run_simulation(silent=True)
    assert success, "MODFLOW 6 did not terminate normally:\n" + "\n".join(buff[-20:])
    return time.time() - t0


t_base = run(sim_base)
print(f"base model: {t_base:4.1f} s")

### Where is the unsaturated zone?

VSF needs cells in the unsaturated zone. Compare the thickness of layer 1 with the depth from the land surface to the water table at the end of the steady-state period. A negative depth means the head is above the land surface; the base model has no surface drain, so heads can rise above it.

In [ ]:
dis = gwf_base.dis
top = dis.top.array
botm = dis.botm.array
idomain = dis.idomain.array
thickness1 = top - botm[0]

# end of each stress period, in days
period_ends = np.cumsum([p[0] for p in sim_base.tdis.perioddata.get_data()])
_, wt_base = vh.read_water_tables(base_ws, vsf=False, times=period_ends)
depth_to_wt = top - wt_base[0]

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]], figsize=(width, 0.55 * width), layout="constrained"
    )
    for letter, array, label in [
        ("A", thickness1, "Thickness of layer 1 (m)"),
        ("B", depth_to_wt, "Depth to the water table (m)"),
    ]:
        ax = axd[letter]
        pmv = flopy.plot.PlotMapView(model=gwf_base, ax=ax)
        pc = pmv.plot_array(array)
        fig.colorbar(pc, ax=ax, shrink=0.8, label=label)
        ax.set_aspect("equal")
    flopy.plot.styles.heading(ax=axd["A"], letter="A", heading="Layer 1 thickness")
    flopy.plot.styles.heading(ax=axd["B"], letter="B", heading="Depth to water table")

depth = depth_to_wt.compressed()
print(
    f"layer 1 is {thickness1.min():.1f} to {thickness1.max():.1f} m thick; "
    f"depth to the water table: median {np.median(depth):.2f} m, "
    f"5-95% {np.percentile(depth, 5):.2f} to {np.percentile(depth, 95):.2f} m"
)

**What to look for.** Layer 1 is a single cell 2 to 8 m thick (A), while the water table is only a few tenths of a meter below the land surface in much of the valley, above it in the north, and at most about 2.5 m below it in the southeast (B). The whole unsaturated zone is therefore **inside one cell**. Richards' equation solves one pressure head per cell, so with this grid there is nothing to resolve. We first confirm that, then fix it.

## Step 1 &mdash; The VSF package

Adding VSF to a model changes four things, whatever the grid:

1. **The package.** VSF is switched on for the cells (`iunsat`), with the soil model and its parameters: `porosity`, the residual saturation `satres`, and for van Genuchten `alphavgn` and `nvgn`. It also saves the pressure head and the saturation next to the head file.
2. **Non-convertible cells.** VSF replaces the convertible-cell (water table) logic of NPF, so `icelltype` and `iconvert` are 0. The soil curves supply the storage above the water table.
3. **Newton.** Richards' equation is strongly nonlinear, so the model uses the Newton formulation with under-relaxation.
4. **A robust solver.** The `COMPLEX` setting, more iterations, and **backtracking**, which shortens a Newton step that makes the residual worse. Without backtracking the steady-state solution of the 0.5 m refined model did not converge.

The package reads one set of soil curves. The curves give the saturation and the relative permeability as a function of the pressure head, and are what makes the unsaturated zone different from the saturated zone.

In [ ]:
soil_name = "loam"  # try "sand" or "clay"
psi = -np.linspace(0.0, 3.0, 300)  # pressure head, m (drier downward on the plot)

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]], figsize=(width, 0.45 * width), layout="constrained"
    )
    for name, soil in SOILS.items():
        lw = 2.0 if name == soil_name else 0.8
        axd["A"].plot(vh.van_genuchten_saturation(psi, soil), -psi, lw=lw, label=name)
        axd["B"].plot(vh.van_genuchten_kr(psi, soil), -psi, lw=lw, label=name)
    for letter, label in [("A", "Saturation (-)"), ("B", "Relative permeability (-)")]:
        axd[letter].set_ylim(3.0, 0.0)
        flopy.plot.styles.xlabel(ax=axd[letter], label=label)
    flopy.plot.styles.ylabel(ax=axd["A"], label="Suction -ψ (m)")
    flopy.plot.styles.heading(ax=axd["A"], letter="A", heading="Saturation")
    flopy.plot.styles.heading(ax=axd["B"], letter="B", heading="Relative permeability")
    flopy.plot.styles.graph_legend(ax=axd["B"], loc="center right")

The curves are drawn for three standard soils; the thick line is the soil used in this notebook. Just above the water table (suction near 0) every soil is nearly saturated and conducts at its full conductivity. That near-saturated band is the **capillary fringe**. Moving up, the soil dries and the relative permeability falls much faster than the saturation.

`add_vsf()` makes the four changes to a model. This is all the VSF-specific input there is.

In [ ]:
def add_vsf(gwf, soil_name="loam"):
    """Switch on Richards flow in a loaded groundwater flow model."""
    soil = SOILS[soil_name]

    # 2. non-convertible cells: VSF supplies the storage above the water table
    gwf.npf.icelltype.set_data(0)
    gwf.sto.iconvert.set_data(0)

    # 1. the VSF package, with the soil curves and the extra output files
    flopy.mf6.ModflowGwfvsf(
        gwf,
        ivarsat=1,
        storage_scheme="chord-slope",
        kr_averaging="arithmetic",
        soil_model="VanGenuchten",
        porosity=soil["porosity"],
        satres=soil["satres"],
        alphavgn=soil["alpha"],
        nvgn=soil["n"],
        phead_filerecord="sv.phd",
        sat_filerecord="sv.sat",
    )
    gwf.oc.saverecord.set_data({0: [("HEAD", "ALL")]})

    # 3. Newton with under-relaxation
    gwf.name_file.newtonoptions = "NEWTON UNDER_RELAXATION"

    # 4. a robust solver for the nonlinear problem
    ims = next(
        p
        for p in gwf.simulation.sim_package_list
        if isinstance(p, flopy.mf6.ModflowIms)
    )
    ims.complexity = "COMPLEX"
    ims.outer_maximum = 500
    ims.inner_maximum = 300
    ims.outer_dvclose = 1e-6
    ims.inner_dvclose = 1e-7
    ims.linear_acceleration = "BICGSTAB"
    ims.under_relaxation = "DBD"
    ims.backtracking_number = 20
    ims.backtracking_tolerance = 1.1
    ims.backtracking_reduction_factor = 0.3
    ims.backtracking_residual_limit = 100.0

### Add VSF to the native grid

First do the minimum: add VSF to the base valley with its grid unchanged, and the same `nstp` time steps per year as the base model. We use 4 (set above) and test that choice at the end.

In [ ]:
native_ws = pl.Path("models/synthetic-valley-vsf-native")
sim_native, gwf_native = vh.load_valley(native_ws)
vh.set_substeps(sim_native, nstp)
add_vsf(gwf_native, soil_name)
t_native = run(sim_native)
print(f"VSF on the native grid: {t_native:4.1f} s")

## Step 2 &mdash; Refine the grid in the vertical

Now give the unsaturated zone cells. `refine_top_layer()` splits the top `refine_depth` meters of layer 1 into sublayers of about `target_dz` meters. One transition cell keeps the rest of layer 1 down to its original bottom, and the layers below are unchanged. We keep the refinement modest: 0.5 m cells over the top 2 m is 4 sublayers, so the model has 4 more layers and runs in tens of seconds.

Changing the layers affects everything that is defined per layer. This is the part of adding VSF that is most work, and it is the same for any model:

| Package | What changes |
|---|---|
| DIS | layer 1 becomes `n_ref` thin sublayers and a transition cell; the layers below move down by `n_ref` |
| IC, NPF, STO | arrays are copied from layer 1 to every new cell; the lake's huge vertical conductivity ratio (about 2e6) is capped, since in thin cells it makes the nonlinear solve fail |
| RCH | no change: recharge still goes to the top cell, and now has to cross the unsaturated zone |
| RIV, GHB | the cell moves to the sublayer that contains the stage or head. A head-dependent boundary in a dry cell does nothing, so it must be in a cell that is wet |
| WEL | the layer numbers shift by `n_ref` |

In [ ]:
target_dz = 0.5  # m; smaller resolves the unsaturated zone better but runs slower
refine_depth = 2.0  # m of layer 1, measured from the land surface

refined_ws = pl.Path("models/synthetic-valley-vsf-refined")
sim_ref, gwf_ref = vh.load_valley(refined_ws)
n_ref = vh.refine_top_layer(sim_ref, target_dz, refine_depth)
vh.set_substeps(sim_ref, nstp)
add_vsf(gwf_ref, soil_name)
t_ref = run(sim_ref)
print(
    f"{n_ref} refined sublayers of ~{target_dz} m: {gwf_ref.dis.nlay.array} layers; "
    f"{t_ref:4.1f} s"
)

### What the unsaturated zone looks like

Read the saturation at the end of the steady-state period and draw it along a north-south section through column 13, together with the cell edges. Cells that are not fully saturated are above the water table.

In [ ]:
col_xs = 12  # column of the section, counted from 0


def end_of_steady_state(ws, name="saturation"):
    return flopy.utils.HeadFile(str(ws / "sv.sat"), text=name).get_data(idx=0)


sat_native = end_of_steady_state(native_ws)
sat_ref = end_of_steady_state(refined_ws)

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A"], ["B"]], figsize=(width, 0.8 * width), layout="constrained", sharex=True
    )
    for letter, gwf, sat, heading in [
        ("A", gwf_native, sat_native, "Native grid"),
        ("B", gwf_ref, sat_ref, "Refined grid"),
    ]:
        ax = axd[letter]
        xs = flopy.plot.PlotCrossSection(model=gwf, line={"column": col_xs}, ax=ax)
        pc = xs.plot_array(np.ma.masked_greater(sat, 1.0e29), vmin=0.4, vmax=1.0)
        xs.plot_grid(lw=0.3, color="0.3")
        ax.set_ylim(-2.0, 6.5)
        flopy.plot.styles.heading(ax=ax, letter=letter, heading=heading)
        flopy.plot.styles.ylabel(ax=ax, label="Elevation (m)")
    flopy.plot.styles.xlabel(ax=axd["B"], label="Distance along the section (m)")
    fig.colorbar(pc, ax=list(axd.values()), shrink=0.8, label="Saturation (-)")

**What to look for.** On the native grid (A) the top cell is saturated everywhere: the unsaturated zone is inside it, so VSF has nothing to resolve. On the refined grid (B) the unsaturated zone appears. In the north the water table is at the land surface and every cell is saturated, but toward the south (right), where the water table is 1 to 2 m below the land surface, the top cells are only partly saturated, and the saturation rises to 1 just above the water table. That near-saturated band is the capillary fringe.

## Step 3 &mdash; What does it change?

Compare the three models at the end of each stress period: the base model, VSF on the native grid, and VSF on the refined grid. Differences are small compared with the water table itself, so plot the **difference from the base model**. The first period is the steady state; pumping starts in the second stress period.

In [ ]:
tw, wt_native = vh.read_water_tables(native_ws, vsf=True, times=period_ends)
_, wt_ref = vh.read_water_tables(refined_ws, vsf=True, times=period_ends)

year = (period_ends - period_ends[0]) / 365.25

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]], figsize=(width, 0.5 * width), layout="constrained"
    )
    ax = axd["A"]
    for label, wt in [("VSF, native grid", wt_native), ("VSF, refined grid", wt_ref)]:
        d = (wt - wt_base).filled(np.nan).reshape(len(year), -1)
        (line,) = ax.plot(year, np.nanmean(d, axis=1), marker="o", ms=3, label=label)
        ax.fill_between(
            year,
            np.nanpercentile(d, 5, axis=1),
            np.nanpercentile(d, 95, axis=1),
            color=line.get_color(),
            alpha=0.2,
        )
    ax.axhline(0.0, color="0.6", lw=0.6)
    flopy.plot.styles.xlabel(ax=ax, label="Years since the end of the steady state")
    flopy.plot.styles.ylabel(ax=ax, label="Water table minus base model (m)")
    flopy.plot.styles.heading(ax=ax, letter="A", heading="Whole valley, mean and 5-95%")
    flopy.plot.styles.graph_legend(ax=ax, loc="upper left")

    ax = axd["B"]
    diff = wt_ref[0] - wt_base[0]
    pmv = flopy.plot.PlotMapView(model=gwf_base, ax=ax)
    pc = pmv.plot_array(diff)
    fig.colorbar(pc, ax=ax, shrink=0.8, label="Refined minus base (m)")
    ax.set_aspect("equal")
    flopy.plot.styles.heading(ax=ax, letter="B", heading="Steady state, map")

for name, wt in [("native", wt_native), ("refined", wt_ref)]:
    d = wt - wt_base
    print(
        f"{name:8s} mean difference {d.mean():+.3f} m, largest {np.abs(d).max():.3f} m"
    )

**What to look for.** Without a refined grid, VSF gives back the base model: the water table is within a few millimeters of the saturated answer on average (A, blue), because every cell is fully saturated and Richards' equation reduces to the saturated flow equation. The refined grid (orange) puts the water table about 4 cm higher on average, and up to 5 cm higher in the steady state (B). It also responds differently when the stresses change. In the dry year at 13 years, when recharge falls by about 40%, the refined water table falls much less than the base model's, by 0.19 m on average: the soil above the water table holds water and delivers it with a delay, storage that the base model leaves out. In this valley the unsaturated zone is thin, so the effect on the water table is modest; the gain is that the model now **simulates** the unsaturated zone.

## Step 4 &mdash; How fine, and how many steps?

The refinement and the time step are the two numbers the VSF model adds. Rerun the refined model with a thicker and a thinner `target_dz` and with a single time step per year, and compare the water table with the model above. This runs three more models, which takes about a minute and a half.

In [ ]:
cases = {
    "dz 1.0 m, 4 steps": (1.0, 4),
    "dz 0.25 m, 4 steps": (0.25, 4),
    "dz 0.5 m, 1 step": (0.5, 1),
}
print(f"{'case':20s} {'run time':>9s} {'mean diff':>10s} {'largest':>8s}")
for name, (dz, steps) in cases.items():
    ws = pl.Path("models/synthetic-valley-vsf-sensitivity")
    sim, gwf = vh.load_valley(ws)
    vh.refine_top_layer(sim, dz, refine_depth)
    vh.set_substeps(sim, steps)
    add_vsf(gwf, soil_name)
    seconds = run(sim)
    _, wt = vh.read_water_tables(ws, vsf=True, times=period_ends)
    d = wt - wt_ref
    print(f"{name:20s} {seconds:7.1f} s {d.mean():+9.3f}  {np.abs(d).max():7.3f}")

**What to look for.** The cell thickness matters little once the zone is resolved: 0.5 m and 0.25 m cells agree within 2 cm, while 1 m cells are off by up to 4 cm in places, as large as the effect of the refinement itself. The time step matters more: one step per year differs by up to 9 cm, more than the typical effect of the refinement (about 4 cm). Thinner cells mostly cost run time, but too long a time step changes the answer. Check both for your own model.

## Recap

To put the VSF package into the valley model you:

- **refined the top layer** so the unsaturated zone has cells, and moved recharge, the river, the coast, and the wells onto the new layers. Without the refinement VSF returned the saturated answer;
- **added the VSF package** with a soil model and its parameters, and made the cells non-convertible;
- **switched to Newton with a robust solver** and **shortened the time step**; and
- checked the **cell thickness** and **time step** against the water table.

What this notebook leaves out: the **land surface**. UZF rejects infiltration the soil cannot take and passes it on as runoff with MVR. With recharge going straight into the top cell, the VSF model has no such limit, and the base model's water table is above the land surface in many cells. A surface boundary for VSF (such as the seepage-face package VSSP) would handle that. Evapotranspiration, which the VSET package represents as root-water uptake, and a soil that varies in space, which the per-cell VSF arrays allow, are the other next steps.